# 02 — A BB84 session, step by step
Every number below comes from `qll.qkd`; change `distance_m` to see the classical time move.

In [ ]:
import numpy as np
from qll.qkd.bb84 import run_bb84
from qll.constants.astro import EARTH_MARS_MIN_M
r = run_bb84(40000, channel_error=0.03, allow_pseudo=True, distance_m=EARTH_MARS_MIN_M)
print(f'sent {r.n_sent}, sifted {r.n_sifted}, QBER {r.qber:.3f}, secret fraction {r.secret_fraction_per_sifted:.3f}')
print(f'final key {r.secret_bits} bits; classical time {r.classical_time_s/60:.1f} min (Mars min)')

## Intercept–resend: 25 % QBER, zero key

In [ ]:
e = run_bb84(40000, intercept_resend=True, allow_pseudo=True)
print(f'QBER {e.qber:.3f} -> secret bits {e.secret_bits}')

## Rate versus QBER and the 11 % threshold

In [ ]:
import matplotlib.pyplot as plt
from qll.qkd.key_rate import bb84_rate_per_sifted_bit, bb84_qber_threshold
Q = np.linspace(0, 0.15, 200)
plt.plot(Q*100, [bb84_rate_per_sifted_bit(q) for q in Q]); plt.axvline(bb84_qber_threshold()*100, ls=':', color='k')
plt.xlabel('QBER (%)'); plt.ylabel('secret bits per sifted bit'); plt.show()

## Which protocol wins at which loss

In [ ]:
from qll.qkd.decoy_state import decoy_rate_per_pulse
from qll.qkd.twin_field import twin_field_rate_per_pulse
from qll.qkd.plob_bound import plob_bits_per_use
eta = np.logspace(-8, -0.1, 300)
plt.loglog(eta, [plob_bits_per_use(e) for e in eta], 'k', label='PLOB')
plt.loglog(eta, [decoy_rate_per_pulse(e) for e in eta], label='decoy BB84')
plt.loglog(eta, [twin_field_rate_per_pulse(e) for e in eta], label='twin-field')
plt.xlabel('transmittance'); plt.ylabel('bits per pulse'); plt.legend(); plt.show()